# 04 - End-to-End Production Training

Runs the complete customer-retention pipeline on the **real Kaggle Telco dataset**
and exports the artefacts the Streamlit app consumes:

- churn model + metrics (`models/`)
- scored, CLV-enriched, segmented customer table (`data/processed/`)
- `retention_decisions.csv` powering the ROI simulator

All paths are built with `os.path.join`. The notebook reuses the project modules so
the notebook, scripts and Streamlit app share identical logic.

In [1]:
import json
import os
import sys

import pandas as pd


def find_project_root(start: str) -> str:
    current = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(current, "src")) and os.path.isdir(
            os.path.join(current, "config")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError("Could not locate the project root (src/ + config/).")
        current = parent


PROJECT_ROOT = find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System


In [2]:
from src.data_pipeline.load_dataset import load_telco_data, telco_raw_path
from src.data_pipeline.feature_engineering import engineer_features, prepare_model_data
from src.churn_model.churn_classifier import ChurnClassifier
from src.churn_model.shap_explainer import ShapExplainer
from src.survival_analysis.survival_model import SurvivalAnalyzer
from src.uplift_modeling.uplift_model import UpliftModeler
from src.clv_segmentation.clv_calculator import CLVCalculator
from src.clv_segmentation.customer_segments import CustomerSegmenter

## 1. Load and validate the real dataset

In [3]:
print("Raw Kaggle file present:", os.path.exists(telco_raw_path()))

# A randomized retention-offer flag is simulated for the Telco book of business so
# the ROI simulator has per-customer uplift to work with. Genuine causal uplift is
# validated on the real Hillstrom experiment in notebook 03.
raw_df = load_telco_data(add_synthetic_treatment=True)

required = {"customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn", "treatment"}
missing = required.difference(raw_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = engineer_features(raw_df)
df["tenure"] = df["tenure"].clip(lower=1)
processed_path = os.path.join(PROCESSED_DATA_DIR, "training_snapshot.csv")
df.to_csv(processed_path, index=False)
print(f"Training rows: {len(df):,}")
print(f"Saved snapshot: {processed_path}")

Raw Kaggle file present: True


Training rows: 7,043
Saved snapshot: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\data\processed\training_snapshot.csv


## 2. Train and evaluate the churn model

In [4]:
classifier, metrics, X_test, y_test = ChurnClassifier.train_and_evaluate(df)
print(pd.Series(metrics).round(4).to_string())
if metrics["auc_roc"] < 0.70:
    print("Warning: ROC-AUC is below the demo target; investigate data quality and calibration.")

classifier.save_model(os.path.join(MODELS_DIR, "churn_lgbm_model.pkl"))
with open(os.path.join(MODELS_DIR, "churn_metrics.json"), "w", encoding="utf-8") as fh:
    json.dump(metrics, fh, indent=2)
print("Saved churn model and metrics.")

The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.


accuracy     0.8006
precision    0.6504
recall       0.4792
f1           0.5518
auc_roc      0.8347
auc_pr       0.6185
Saved churn model and metrics.


In [5]:
X_all, _ = prepare_model_data(df)
df["churn_probability"] = classifier.predict(X_all)

explainer = ShapExplainer.generate_explanations(
    classifier, X_test, save_dir=os.path.join(PROJECT_ROOT, "docs", "diagrams")
)
display(explainer.get_feature_importance_shap().head(15))

LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray


,feature,mean_abs_shap
0,Contract,0.885200
1,tenure,0.310107
2,MonthlyCharges,0.301054
3,avg_monthly_charge,0.152061
4,StreamingTV,0.143957
5,PaperlessBilling,0.121954
6,PaymentMethod,0.087112
7,TotalCharges,0.077137
8,OnlineSecurity,0.074215
9,TechSupport,0.069840


## 3. Survival and uplift models

In [6]:
survival = SurvivalAnalyzer.run_full_analysis(df)
display(survival.get_cox_summary()[["coef", "exp(coef)", "p"]].round(4))

--- Fitting Kaplan-Meier (overall) ---
  Median survival time (overall): inf months
--- Fitting Kaplan-Meier by Contract ---
  Month-to-month: median survival = 37.0 months
  One year: median survival = inf months
  Two year: median survival = inf months
--- Fitting Cox PH model ---



Cox PH Model Summary:
                             coef  exp(coef)  se(coef)  coef lower 95%  coef upper 95%  exp(coef) lower 95%  exp(coef) upper 95%  cmp to          z             p    -log2(p)
covariate                                                                                                                                                                    
SeniorCitizen           -0.072315   0.930238  0.054882       -0.179881        0.035252             0.835369             1.035880     0.0  -1.317647  1.876218e-01    2.414101
MonthlyCharges           0.005906   1.005924  0.002574        0.000861        0.010951             1.000862             1.011012     0.0   2.294526  2.176033e-02    5.522156
Contract_encoded        -1.014441   0.362605  0.089739       -1.190327       -0.838555             0.304122             0.432335     0.0 -11.304304  1.249402e-29   96.014677
InternetService_encoded  0.597197   1.817019  0.075757        0.448717        0.745677             1.566301

[SurvivalAnalyzer] Kaplan-Meier plot saved: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams\kaplan_meier.png


[SurvivalAnalyzer] Cox coefficients plot saved: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams\cox_coefficients.png


,coef,exp(coef),p
covariate,,,
SeniorCitizen,-0.0723,0.9302,0.1876
MonthlyCharges,0.0059,1.0059,0.0218
Contract_encoded,-1.0144,0.3626,0.0000
InternetService_encoded,0.5972,1.8170,0.0000
num_services,-0.3100,0.7335,0.0000
auto_pay,-0.6470,0.5236,0.0000
contract_risk,0.8209,2.2726,0.0000


In [7]:
X_t, y_t = prepare_model_data(df)
treatment_t = df["treatment"].loc[X_t.index]

uplift_model = UpliftModeler()
uplift_model.fit(X_t, treatment_t, y_t)
uplift_model.predict_uplift(X_t)
uplift_segments = uplift_model.get_uplift_segments(X_t, treatment_t, y_t)

# Keep the uplift segment separate so KMeans can own the 'segment' column later.
df["uplift_score"] = uplift_segments["uplift_score"]
df["uplift_segment"] = uplift_segments["segment"]
print("Uplift engine:", "causalml" if uplift_model.using_causalml else "Two-Model")
print(df["uplift_segment"].value_counts().to_string())

[UpliftModeler] Using causalml BaseSClassifier

Uplift engine: causalml
uplift_segment
Sleeping Dog    3751
Persuadable     1761
Lost Cause      1531


## 4. CLV, customer segments and offer economics

In [8]:
df = CLVCalculator.compute_all_clv(df, df["churn_probability"].to_numpy())

segmenter, segmented_df, profiles = CustomerSegmenter.run_segmentation(df)
df = segmented_df
display(profiles[[
    "segment", "label", "count", "pct_of_total", "avg_clv",
    "avg_churn_probability", "dominant_contract", "recommended_action",
]].round(3))

,segment,label,count,pct_of_total,avg_clv,avg_churn_probability,dominant_contract,recommended_action
0,0,Champions,862,12.2,683.38,0.034,Two year,Loyalty programme – upsell & cross-sell
1,1,Price Sensitive,1789,25.4,151.31,0.199,One year,Loyalty programme – upsell & cross-sell
2,2,Champions,2271,32.2,161.35,0.104,Month-to-month,Loyalty programme – upsell & cross-sell
3,3,Price Sensitive,2121,30.1,29.42,0.553,Month-to-month,Value-based discounts – cost-sensitive retention


In [9]:
roi = CLVCalculator().compute_expected_revenue_impact(
    df, df["uplift_score"].to_numpy(), offer_cost=10.0
)
decision_table = df[[
    "customerID", "churn_probability", "uplift_score", "uplift_segment",
    "clv", "segment", "segment_label", "MonthlyCharges",
]].join(roi)
display(decision_table.sort_values("net_value", ascending=False).head(20).round(3))

,customerID,churn_probability,uplift_score,uplift_segment,clv,segment,segment_label,MonthlyCharges,expected_retention_months_gained,revenue_from_retention,roi,net_value
5441,3512-IZIKN,0.020,0.135,Persuadable,778.738,0,Champions,65.30,6.515,132.859,12.286,122.859
4527,7666-WKRON,0.036,0.125,Persuadable,584.185,0,Champions,80.45,3.327,71.265,6.127,61.265
3779,8631-NBHFZ,0.032,0.093,Persuadable,633.320,0,Champions,79.40,2.796,70.335,6.034,60.335
6792,4597-ELFTS,0.021,0.175,Persuadable,286.701,2,Champions,24.90,7.966,57.671,4.767,47.671
5589,7253-UVNDW,0.033,0.110,Persuadable,420.665,2,Champions,54.35,3.194,48.145,3.815,38.145
4272,1043-YCUTE,0.031,0.200,Persuadable,212.354,2,Champions,25.15,6.322,44.033,3.403,34.033
6785,3090-HAWSU,0.062,0.034,Persuadable,471.560,0,Champions,111.60,0.504,33.215,2.322,23.215
5947,7951-QKZPL,0.018,0.071,Persuadable,321.474,2,Champions,24.50,3.821,28.823,1.882,18.823
1254,9220-ZNKJI,0.042,0.023,Persuadable,557.792,0,Champions,88.80,0.526,26.429,1.643,16.429
5992,0687-ZVTHB,0.052,0.059,Persuadable,363.104,1,Price Sensitive,72.45,1.074,21.563,1.156,11.563


In [10]:
decision_path = os.path.join(PROCESSED_DATA_DIR, "retention_decisions.csv")
decision_table.to_csv(decision_path, index=False)
scored_path = os.path.join(PROCESSED_DATA_DIR, "telco_churn_scored.csv")
df.to_csv(scored_path, index=False)

print("Saved retention decisions to", decision_path)
print("Profitable offers at $10 cost:", int((decision_table['net_value'] > 0).sum()))
print("Total net value:", round(decision_table['net_value'].clip(lower=0).sum(), 2))

Saved retention decisions to C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\data\processed\retention_decisions.csv
Profitable offers at $10 cost: 13
Total net value: 451.33


## Production checklist

- Replace the simulated offer assignment with a real randomized retention campaign.
- Keep the test split untouched until final evaluation.
- Store the training snapshot, model files, metrics and feature list together.
- Validate uplift on randomized campaign data before spending money on offers.
- Add scheduled retraining, drift monitoring and approval before deploying.